# Capstone notebook (M7): <hazard> - <event name, place, dates>

**Author:** <your name>
**Course:** Application of geoinformatics to natural hazards mapping and monitoring, lab 7
**Workflow reproduced:** lab 4 (flood extent, Sentinel-1) **or** lab 6 (burn severity, dNBR) - keep one section, delete the other.

> How to use this template: run the setup cell, fill in the parameters, run the workflow cells, then replace every `<...>` in the text cells with your own words. A colleague must be able to run all cells top to bottom after `ee.Initialize(project=...)` and obtain the same map and numbers.


In [ ]:
# --- Setup: run once per Colab session ---------------------------------
# geemap is installed on demand in Colab; locally install it with pip/conda.
try:
    import geemap
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'geemap'], check=True)
    import geemap

import ee

PROJECT = 'ee-your-project'   # <-- the Cloud project id from lab 1 (id, not name)

ee.Authenticate()             # opens a browser window - sign in once per session
ee.Initialize(project=PROJECT)
print(ee.String('Hello from Earth Engine').getInfo())


## 1. Event description

<Two or three paragraphs: what happened, where and when. Trigger (rainfall, drought and heat, earthquake), extent reported by authorities, Copernicus EMS activation number or EFFIS / FIRMS evidence, why this event matters. Cite your sources.>

Default examples in this template:

- **Flood:** Nysa Klodzka valley, south-west Poland, September 2024 (Copernicus EMS activation EMSR756; peak 15-17 September after storm Boris).
- **Fire:** Landiras fire, Gironde, France, 12-25 July 2022 (about 14 000 ha of maritime pine; a second fire reignited nearby on 9 August 2022).


## 2. Data

| Dataset | Earth Engine id | Role |
|---|---|---|
| Sentinel-1 GRD | `COPERNICUS/S1_GRD` | VH backscatter in dB, before/after (flood) |
| Sentinel-2 L2A | `COPERNICUS/S2_SR_HARMONIZED` | pre/post reflectance, SCL cloud mask (fire) |
| Cloud Score+ | `GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED` | optional per-pixel cloud mask (`cs_cdf >= 0.6`) |
| JRC Global Surface Water | `JRC/GSW1_4/GlobalSurfaceWater` | permanent water mask (`seasonality >= 10`) |
| NASADEM | `NASA/NASADEM_HGT/001` | slope mask (< 5 degrees) |

<Say which of these you used, the acquisition dates found in the collections (print them below), orbit direction, and where the AOI comes from (drawn, EMS perimeter, EFFIS polygon).>


## 3. Parameters

All parameters in one place: a reader changes them here and nowhere else.


In [ ]:
# --- Common ---------------------------------------------------------------
EXPORT_CRS = 'EPSG:2180'       # Poland; use the national CRS of your event (France: EPSG:2154)
DRIVE_FOLDER = 'gee'

# --- Flood (lab 4) --------------------------------------------------------
FLOOD_BBOX = [17.05, 50.38, 17.75, 50.62]              # west, south, east, north
BEFORE_START, BEFORE_END = '2024-08-20', '2024-09-10'
AFTER_START, AFTER_END = '2024-09-16', '2024-09-25'
POLARISATION = 'VH'
ORBIT_PASS = 'DESCENDING'
SMOOTH_RADIUS_M = 50
DIFF_THRESHOLD_DB = -3.0
PERMANENT_WATER_MONTHS = 10
MAX_SLOPE_DEG = 5
MIN_CONNECTED_PIXELS = 8

# --- Fire (lab 6) ---------------------------------------------------------
FIRE_BBOX = [-0.70, 44.38, -0.30, 44.65]
PRE_START, PRE_END = '2022-06-15', '2022-07-10'
POST_START, POST_END = '2022-07-26', '2022-08-08'
MAX_CLOUD_PCT = 40
USE_CLOUD_SCORE_PLUS = False
CS_PLUS_THRESHOLD = 0.6
CLASS_BREAKS = [-250, -100, 100, 270, 440, 660]        # USGS, dNBR x 1000
CLASS_NAMES = ['enhanced regrowth, high', 'enhanced regrowth, low', 'unburned',
               'low severity', 'moderate-low severity', 'moderate-high severity',
               'high severity']
CLASS_PALETTE = ['7a8737', 'acbe4d', '0ae042', 'fff70b', 'ffaf38', 'ff641b', 'a41fd6']


## 4. Method

<Describe the chain in your own words, with the reasons for each step: why one orbit direction, why a speckle filter, why -3 dB (or ratio 1.25), why the JRC and slope masks; or: why SCL per-pixel masking, why NBR uses B8 and B12, why dNBR x 1000 and the USGS breaks. Reference the UN-SPIDER recommended practice and the lecture.>

### 4a. Flood extent from Sentinel-1 change detection (lab 4)

Delete this subsection if your capstone is the fire workflow.


In [ ]:
# Flood workflow - same code as examples/python/03_flood_s1_change_detection.py
aoi = ee.Geometry.Rectangle(FLOOD_BBOX)

s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', POLARISATION))
      .filter(ee.Filter.eq('orbitProperties_pass', ORBIT_PASS))
      .filterBounds(aoi)
      .select(POLARISATION))

s1_before = s1.filterDate(BEFORE_START, BEFORE_END)
s1_after = s1.filterDate(AFTER_START, AFTER_END)
print('before scenes:', s1_before.size().getInfo())
print('after scenes :', s1_after.size().getInfo())
print('after dates  :', s1_after.aggregate_array('system:index').getInfo())

before_f = s1_before.mosaic().clip(aoi).focal_mean(SMOOTH_RADIUS_M, 'circle', 'meters')
after_f = s1_after.mosaic().clip(aoi).focal_mean(SMOOTH_RADIUS_M, 'circle', 'meters')

difference = after_f.subtract(before_f).rename('diff_dB')
flooded_raw = difference.lt(DIFF_THRESHOLD_DB).rename('flooded')

permanent_water = (ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
                   .select('seasonality').gte(PERMANENT_WATER_MONTHS))
slope = ee.Terrain.slope(ee.Image('NASA/NASADEM_HGT/001').select('elevation'))

flooded = (flooded_raw
           .updateMask(flooded_raw)
           .updateMask(permanent_water.unmask(0).Not())
           .updateMask(slope.lt(MAX_SLOPE_DEG))
           .updateMask(flooded_raw.connectedPixelCount(25).gte(MIN_CONNECTED_PIXELS)))

area_ha = (flooded.multiply(ee.Image.pixelArea()).divide(1e4)
           .reduceRegion(ee.Reducer.sum(), aoi, 10, maxPixels=1e10)
           .get('flooded').getInfo())
print('flooded area [ha]: %.0f' % area_ha)


In [ ]:
# Flood map
m = geemap.Map()
m.centerObject(aoi, 11)
m.addLayer(before_f, {'min': -25, 'max': 0}, 'before ' + POLARISATION)
m.addLayer(after_f, {'min': -25, 'max': 0}, 'after ' + POLARISATION)
m.addLayer(difference, {'min': -10, 'max': 10, 'palette': ['red', 'white', 'blue']}, 'difference dB', False)
m.addLayer(flooded, {'palette': ['0000FF']}, 'flooded')
m.addLayer(aoi, {'color': 'red'}, 'AOI', False)
m


In [ ]:
# Flood sensitivity table (threshold vs area) - the uncertainty of your map
import pandas as pd

rows = []
for thr in (-2.0, -3.0, -4.0):
    for radius in (30, 50, 100):
        b = s1_before.mosaic().clip(aoi).focal_mean(radius, 'circle', 'meters')
        a = s1_after.mosaic().clip(aoi).focal_mean(radius, 'circle', 'meters')
        f = (a.subtract(b).lt(thr)
             .updateMask(permanent_water.unmask(0).Not())
             .updateMask(slope.lt(MAX_SLOPE_DEG)))
        ha = (f.multiply(ee.Image.pixelArea()).divide(1e4)
              .reduceRegion(ee.Reducer.sum(), aoi, 10, maxPixels=1e10)
              .values().get(0).getInfo())
        rows.append({'threshold_dB': thr, 'radius_m': radius, 'flooded_ha': round(ha)})

sens = pd.DataFrame(rows).pivot(index='threshold_dB', columns='radius_m', values='flooded_ha')
sens


### 4b. Burn severity from dNBR (lab 6)

Delete this subsection if your capstone is the flood workflow.


In [ ]:
# Fire workflow - same code as examples/python/04_dnbr_burn_severity.py
aoi = ee.Geometry.Rectangle(FIRE_BBOX)


def mask_scl(img):
    keep = img.select('SCL').remap([4, 5, 6, 11], [1, 1, 1, 1], 0)
    return img.updateMask(keep)


def mask_cs_plus(img):
    return img.updateMask(img.select('cs_cdf').gte(CS_PLUS_THRESHOLD))


def s2_masked(start, end):
    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterDate(start, end).filterBounds(aoi)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', MAX_CLOUD_PCT)))
    if USE_CLOUD_SCORE_PLUS:
        cs_plus = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
        return s2.linkCollection(cs_plus, ['cs_cdf']).map(mask_cs_plus)
    return s2.map(mask_scl)


s2_before = s2_masked(PRE_START, PRE_END)
s2_after = s2_masked(POST_START, POST_END)
print('pre scenes :', s2_before.size().getInfo())
print('post scenes:', s2_after.size().getInfo())

pre = s2_before.median().clip(aoi)
post = s2_after.median().clip(aoi)


def nbr(img):
    return img.normalizedDifference(['B8', 'B12']).rename('NBR')


dnbr = nbr(pre).subtract(nbr(post)).multiply(1000).rename('dNBR')

severity = ee.Image.constant(0).rename('severity')
for brk in CLASS_BREAKS:
    severity = severity.add(dnbr.gt(brk))
severity = severity.updateMask(dnbr.mask()).toByte()

groups = (ee.Image.pixelArea().divide(1e4).addBands(severity)
          .reduceRegion(ee.Reducer.sum().group(groupField=1, groupName='class'),
                        aoi, 20, maxPixels=1e10)
          .get('groups').getInfo())

import pandas as pd
areas = pd.DataFrame([{'class': g['class'], 'name': CLASS_NAMES[g['class']],
                       'ha': round(g['sum'])} for g in groups])
print('total ha:', areas['ha'].sum())
areas


In [ ]:
# Fire map
rgb = {'min': 0, 'max': 3000, 'bands': ['B4', 'B3', 'B2']}
m = geemap.Map()
m.centerObject(aoi, 11)
m.addLayer(pre, rgb, 'pre-fire true colour')
m.addLayer(post, rgb, 'post-fire true colour')
m.addLayer(dnbr, {'min': -500, 'max': 1300, 'palette': ['white', 'black']}, 'dNBR x 1000', False)
m.addLayer(severity, {'min': 0, 'max': 6, 'palette': CLASS_PALETTE}, 'burn severity (USGS)')
m.add_legend(title='dNBR severity', labels=CLASS_NAMES, colors=CLASS_PALETTE)
m


In [ ]:
# dNBR histogram over the AOI (checklist: bimodal, unburned peak near 0?)
import matplotlib.pyplot as plt

hist = (dnbr.reduceRegion(ee.Reducer.fixedHistogram(-500, 1300, 36), aoi, 100, maxPixels=1e10)
        .get('dNBR').getInfo())
lo = [h[0] for h in hist]
n = [h[1] for h in hist]
plt.bar(lo, n, width=50, align='edge')
for brk in CLASS_BREAKS:
    plt.axvline(brk, color='red', linestyle='--', linewidth=0.8)
plt.xlabel('dNBR x 1000')
plt.ylabel('pixels (100 m)')
plt.title('dNBR histogram')
plt.show()


## 5. Results

<Report the numbers from the cells above and interpret them: flooded area in hectares and how it changes across the sensitivity table (factor < 2 between thresholds?), or hectares per severity class and the share of high severity inside the official perimeter. Describe what the map shows: where the flood/fire is, where the map is clearly wrong (fields, shadow, towns). Compare with the Copernicus EMS / GFM / EFFIS product in one paragraph: agreement, disagreement, and what you compared against.>


In [ ]:
# Optional: export the result to Drive (asynchronous; check the Tasks tab or task.status())
# Flood:
# task = ee.batch.Export.image.toDrive(image=flooded.unmask(0).toByte(),
#         description='flood_<event>_<date>', folder=DRIVE_FOLDER,
#         scale=10, region=aoi, crs=EXPORT_CRS, maxPixels=1e10)
# Fire:
# task = ee.batch.Export.image.toDrive(image=severity.unmask(255),
#         description='dnbr_severity_<event>_<date>', folder=DRIVE_FOLDER,
#         scale=20, region=aoi, crs=EXPORT_CRS, maxPixels=1e10)
# task.start(); print(task.status()['state'])


## 6. Limitations

<Be specific to your event. Typical points:>

- **Clouds / weather:** SAR sees through clouds, but wind roughens water (missed flood); optical pre/post composites may still contain cloud shadow after the SCL mask.
- **Phenology:** pre and post from different seasons produce false "low severity" or false change over fields; harvested fields look like burn or flood.
- **Resolution:** 10 m (S1 GRD, S2 B8) vs 20 m (B12, SCL) vs 30 m (NASADEM, JRC); small streams, roads and urban flooding are below the pixel size; speckle filter smooths edges.
- **Thresholds:** -3 dB / ratio 1.25 and the USGS dNBR breaks were not calibrated for your area; report them as proposed values and show the sensitivity.
- **Masks:** permanent-water and slope masks remove real flood on steep valley floors or in reservoirs; connected-pixel filter drops small real patches.
- **Validation:** what you compared against (EMS, GFM, EFFIS, news, field photos) and what you could not check.

## 7. References

- UN-SPIDER recommended practices: flood mapping (Sentinel-1) and burn severity mapping (dNBR).
- Lesson pages and lecture 4 / 6 / 7 of the course.
- <Copernicus EMS activation / EFFIS report / papers you used.>
